# Fit Cognitive Models and Compute -LL

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
# import packages
import os
import numpy as np
import numpy.random as rng
import pandas as pd
from scipy.optimize import minimize, differential_evolution

In [ ]:
# import functions and configurations
from helper_functions import load_data, num_parameters, get_bounds, nll_truncnorm, truncnorm_logpdf, fit_participant
from models import model_CAM, model_GCM
from configurations import DOMAIN_CONFIG, OPTIM_CONFIG

In [56]:
# avoid scientific notation for outputs
np.set_printoptions(suppress=True)

## Example: Mammals

In [59]:
# Define upper limit for predicted criterion estimates
DOMAIN  = "Mammals"
N_CUES  = DOMAIN_CONFIG[DOMAIN]["n_dim"]
UB_CRIT = DOMAIN_CONFIG[DOMAIN]["ub"]

In [60]:
# Load data
cues, ex_cues, ex_crit, ex_ids, test_ids, data = load_data(DOMAIN)

[Mammals] test items: 68, exemplars: 12, participants: 48


In [10]:
def init_pars_CAM(n_dim):

    # Init weight parameters
    w = np.zeros(n_dim+1)

    # Intercept
    w[0]   = rng.normal(984.65, 300)

    # Dimension weights (Importance)
    w[1:]  = rng.normal(0, 750, size = n_dim)

    sigma  = rng.exponential(1/0.01)

    return np.r_[w.squeeze(), sigma]

In [11]:
def init_pars_GCM(n_dim):

    # Init weight parameters
    c      = rng.exponential(1/0.25)
    w      = rng.dirichlet(np.ones(n_dim), size = 1) * n_dim
    sigma  = rng.exponential(1/0.01)

    return np.r_[c, w.squeeze(), sigma]

In [16]:
def inits(model_name, n_dim):
    
    if model_name == 'CAM':
        return init_pars_CAM(n_dim)

    elif model_name == 'GCM':
        return init_pars_GCM(n_dim)
    
    else:
        raise ValueError(f"Unknown model: {model_name}")

### Fit participants and compute AIC & -LL

In [ ]:
model_name = "GCM"  
model_fn   = model_GCM
k          = num_parameters(model_name, N_CUES)
n_subs     = data.shape[1]
n_trials   = data.shape[0]

# Result containers
all_params    = []
all_trial_nll = np.zeros((n_subs, n_trials))
all_aic_gcm       = np.zeros(n_subs)

for i in range(n_subs):
    x = data[:, i]

    params, total_nll, trial_nll = fit_participant(
        x, model_name, model_fn, cues, ex_cues, ex_crit, n_dim = N_CUES,
                 ub = UB_CRIT,
                 inits_fn = inits, optim_config=OPTIM_CONFIG
    )

    # Sanity check: returned GCM weights sum to n_dim
    if model_name == "GCM":
        assert np.isclose(params[1:1 + N_CUES].sum(), N_CUES)
        assert np.all(params[1:1 + N_CUES] >= 0)

    aic               = 2 * k + 2 * total_nll
    all_params.append(params)
    all_trial_nll[i]  = trial_nll
    all_aic_gcm[i]        = aic

    print(f"  Sub {i+1:>3d}/{n_subs}  NLL ={total_nll:8.2f}  AIC ={aic:8.2f}")

  Sub   1/48  NLL =  519.03  AIC = 1062.06
  Sub   2/48  NLL =  502.12  AIC = 1028.23
